# Kirby: bounded training search, 172 development questions
Use your **existing Colab runtime**, verified rollback report, frozen 256-question plan, and cached model. This run freezes the first **172** development IDs from that plan without resampling or overwriting it. Upload the new `kirby_colab.zip`. If the runtime was deleted, rebuild and verify rollback with the earlier notebooks first.

Add `OPENAI_API_KEY` in Colab's Secrets panel and enable notebook access. The key is passed only to the worker environment, not saved in evidence. The request requires account access to `gpt-6-astra`; there is no fallback to another model.

This makes **at most one paid API request**, only if training search confirms an eligible candidate. Limits remain 4,096 output tokens, 128,000 request bytes, 180 seconds per request, and no retries. It sends bounded public training evidence and development aggregates, never final questions or model weights.

Astra may propose one bounded column scale or abstain. The worker validates its answer, evaluates the original and candidate on the same development questions, then **always rolls back**, including favorable results. There is no released checkpoint or demonstrated final-test improvement.


## 1. Update the existing workspace
This restores notebook variables, installs the optional OpenAI SDK into the existing worker, runs tests, and validates your plan. It does not make an API request. Keep GPU precision unchanged from the verified rollback run.


In [ ]:
from google.colab import files, userdata
from pathlib import Path, PurePosixPath
import hashlib, io, json, os, re, signal, stat, subprocess, sys, uuid, zipfile

# If several workspaces exist, paste the intended backend path here.
BACKEND_PATH = ""
WORKER_PYTHON = ""
if BACKEND_PATH:
    backend = Path(BACKEND_PATH)
else:
    candidates = sorted(Path("/content").glob("kirby-*/backend"))
    candidates = [p for p in candidates if (p / "data" / "plan-dev-256" / "development.json").is_file()]
    if len(candidates) != 1:
        raise RuntimeError("Set BACKEND_PATH to the backend with your 256-question plan. Found: "
                           + str([str(p) for p in candidates])
                           + ". If the runtime was reset, use the main notebook to rebuild the baseline first.")
    backend = candidates[0]
workspace = backend.parent
if not (backend / "data" / "plan-dev-256" / "development.json").is_file():
    raise RuntimeError("This workspace has no 256-question development plan.")
workers = sorted(workspace.glob("worker-venv-*/bin/python"),
                 key=lambda p: p.parent.parent.stat().st_mtime)
worker_python = WORKER_PYTHON or (str(workers[-1]) if workers else "")
if not worker_python or not Path(worker_python).is_file():
    raise RuntimeError("Worker environment missing. Run setup in the main notebook first.")
print("Updating:", backend)
bundle_bytes = globals().pop("KIRBY_BUNDLE_BYTES", None)
if bundle_bytes is None:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload only the newly built kirby_colab.zip.")
    bundle_bytes = next(iter(uploaded.values()))
    del uploaded
with zipfile.ZipFile(io.BytesIO(bundle_bytes)) as archive:
    members = archive.infolist()
    if len(members) > 100 or sum(m.file_size for m in members) > 5_000_000:
        raise ValueError("Unexpected source bundle size.")
    replacements = {}
    for member in members:
        name = PurePosixPath(member.filename)
        allowed = (str(name) in {"backend/README.md", "backend/pyproject.toml"} or
                   (len(name.parts) == 3 and name.parts[0] == "backend" and
                    ((name.parts[1] in {"kirby", "tests"} and name.suffix == ".py") or
                     (name.parts[1] == "notebooks" and name.suffix == ".ipynb") or
                     (name.parts[1] == "docs" and name.suffix == ".md"))))
        if (not allowed or ".." in name.parts or name.is_absolute() or
                "\\" in member.filename or ":" in member.filename or
                stat.S_ISLNK(member.external_attr >> 16) or member.filename in replacements):
            raise ValueError("Unsafe or unexpected source bundle member.")
        replacements[member.filename] = archive.read(member)
    required = {"backend/kirby/astra.py", "backend/kirby/astra_runner.py", "backend/kirby/diagnostics.py",
                "backend/kirby/answer_scores.py", "backend/kirby/evidence.py",
                "backend/kirby/search.py", "backend/kirby/search_runner.py"}
    if not required.issubset(replacements):
        raise ValueError("This ZIP predates the Astra integration.")
for name, payload in replacements.items():
    target = workspace / name
    if target.is_symlink() or not target.resolve().is_relative_to(backend.resolve()):
        raise ValueError("Source destination leaves the backend workspace.")
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(payload)
del bundle_bytes, replacements

def run_setup(command, **kwargs):
    result = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, **kwargs)
    print(result.stdout, end="")
    result.check_returncode()
    return result

def run_worker(command, *, env=None):
    process = subprocess.Popen(command, cwd=backend, env=env, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True)
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        code = process.wait()
    except KeyboardInterrupt:
        process.send_signal(signal.SIGINT)
        try:
            remaining, _ = process.communicate(timeout=30)
        except subprocess.TimeoutExpired:
            process.kill()
            remaining, _ = process.communicate()
        print(remaining, end="")
        raise
    finally:
        process.stdout.close()
    if code:
        raise subprocess.CalledProcessError(code, command)

MODEL_ID = "johnsnowlabs/JSL-MedPhi2-2.7B"
MODEL_REVISION = "b313a242eb211863f7b62624b345b321a3b114ea"
PRECISION = "float32"  # Matches your T4 baseline.
MAX_SECONDS = 3600
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
os.environ.setdefault("HF_HOME", "/content/kirby-hf-cache")
if not os.environ.get("HF_TOKEN"):
    os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
revision = MODEL_REVISION
(workspace / "model-revision.txt").write_text(revision, encoding="utf-8")
plan = backend / "data" / "plan-dev-256"
runs = backend / "runs"
run_setup([worker_python, "-c", "import torch; from transformers import PhiConfig, PhiForCausalLM; assert torch.cuda.is_available(), 'Select a GPU runtime'; print(torch.cuda.get_device_name(0))"], cwd=backend)
run_setup([worker_python, "-m", "pip", "install", "-e", str(backend) + "[astra]"], cwd=backend)
run_setup([worker_python, "-c", "from openai import OpenAI; print('OpenAI SDK ready; no API call made')"], cwd=backend)
run_setup([worker_python, "-m", "unittest", "discover", "-s", "tests", "-v"], cwd=backend)
run_setup([worker_python, "-m", "kirby", "validate", "--manifest", str(plan / "development.json")], cwd=backend)
saved_plan = json.loads((plan / 'development.json').read_text())
if len(saved_plan['question_ids']) < 172 or len(saved_plan['diagnostic_ids']) < 64:
    raise RuntimeError('This search needs at least 172 frozen development IDs and 64 training diagnostic IDs.')
print('This run uses 172 development questions, 8 training screen questions, and 56 separate training confirmation questions.')
print("Ready. Dataset and existing evidence retained.")


## 2. Run the bounded search
The worker screens one column in each of eight evenly spaced layers at 0.9 and 1.1, using eight training questions. It confirms at most two winners on the remaining 56 training questions. Ranking requires positive correct-label log-score and margin changes without losing preferred-correct answers. Prior development-evaluated operations are excluded. One seeded random edit is measured on the same confirmation cohort as a separate control. Astra reviews the top confirmed operation or abstains; at most one candidate is evaluated on 172 development questions. Every edit is restored.

This version uses `astra-search-v1-172-attempt-id.txt`, separate from the earlier experiments. First execution starts a new search and may make one paid API call. Repeated executions display saved evidence. Keep earlier reports: the search uses them to avoid repeating evaluated candidates. The new next-token A/B/C/D score protocol differs from the old label-period metric; compare only within a run.


In [ ]:
# Keep this path empty to select the latest verified rollback run by finish time.
ROLLBACK_REPORT = ""
if ROLLBACK_REPORT:
    verified_rollback = Path(ROLLBACK_REPORT)
else:
    candidates = []
    for path in runs.glob("*/report.json"):
        saved = json.loads(path.read_text(encoding="utf-8"))
        if (saved.get("task") == "controlled_rollback_check" and saved.get("state") == "completed"
                and saved.get("weights_restored") is True and saved.get("outputs_restored") is True):
            candidates.append((saved["finished_at"], path))
    if not candidates:
        raise RuntimeError("Run the controlled rollback notebook first.")
    verified_rollback = max(candidates, key=lambda item: item[0])[1]
print("Verified rollback:", verified_rollback)

# A persistent ID makes rerunning this cell reuse evidence, not another API call.
attempt_file = workspace / "astra-search-v1-172-attempt-id.txt"
if attempt_file.exists():
    ASTRA_ATTEMPT_ID = str(uuid.UUID(attempt_file.read_text(encoding="utf-8").strip()))
else:
    ASTRA_ATTEMPT_ID = str(uuid.uuid4())
    with attempt_file.open("x", encoding="utf-8") as stream:
        stream.write(ASTRA_ATTEMPT_ID)
astra_report_path = runs / ASTRA_ATTEMPT_ID / "report.json"
print("Selected 172-question search attempt:", ASTRA_ATTEMPT_ID)

def display_astra_report():
    if astra_report_path.is_file():
        result = json.loads(astra_report_path.read_text(encoding="utf-8"))
        print(json.dumps({key: result.get(key) for key in
                         ("state", "outcome", "comparison", "weights_restored", "outputs_restored",
                          "candidate_retained", "usage", "error")}, indent=2))
        editor = result.get("editor") or {}
        if editor.get("status") == "request_failed":
            print("Provider diagnostics:", json.dumps({key: editor.get(key) for key in
                ("error_type", "http_status", "provider_code", "provider_parameter",
                 "request_id", "guidance", "usage_known")}, indent=2))
        print("Astra evidence:", astra_report_path)
        if result.get('search'):
            print('Search:', json.dumps(result['search'], indent=2))
        if result.get('development_score_comparison'):
            score_change = result['development_score_comparison']
            print('Development score diagnostic (separate from accuracy):', json.dumps({key: score_change.get(key) for key in
                ('mean_expected_margin_delta', 'mean_expected_log_score_delta', 'preferred_label_changes')}, indent=2))
        for probe in result.get("training_sensitivity", []):
            print("Training probe:", probe["probe_id"], probe.get("state"),
                  "mean correct-label margin change:", (probe.get("comparison") or {}).get("mean_expected_margin_delta"))
        if result.get("candidate_training_comparison"):
            scores = result["candidate_training_comparison"]
            print("Candidate training scores:", json.dumps({key: scores.get(key) for key in
                ("mean_expected_margin_delta", "max_absolute_log_score_delta", "preferred_label_changes")}, indent=2))
            print("Training scores restored:", result.get("answer_scores_restored"))

if astra_report_path.exists():
    print("This attempt already ran. Showing saved evidence; no new API call.")
    display_astra_report()
else:
    worker_env = os.environ.copy()
    try:
        try:
            worker_env["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
        except Exception:
            raise RuntimeError("Add OPENAI_API_KEY in Colab Secrets and enable notebook access.") from None
        if not isinstance(worker_env["OPENAI_API_KEY"], str) or not worker_env["OPENAI_API_KEY"].strip():
            raise RuntimeError("OPENAI_API_KEY is empty.")
        worker_env["OPENAI_API_KEY"] = worker_env["OPENAI_API_KEY"].strip()
        print("Starting bounded training search and 172-question development comparison. At most one paid Astra call; no retries.")
        run_worker([worker_python, "-u", "-m", "kirby", "astra-search",
                    "--manifest", str(plan / "development.json"), "--rollback-report", str(verified_rollback),
                    "--attempt-id", ASTRA_ATTEMPT_ID, "--model", MODEL_ID, "--revision", MODEL_REVISION,
                    "--precision", PRECISION, "--allow-download", "--max-seconds", str(MAX_SECONDS),
                    "--development-count", "172", "--diagnostic-count", "64", "--screening-count", "8",
                    "--layer-count", "8", "--confirm-count", "2", "--calibration-questions", "4",
                    "--editor-model", "gpt-6-astra", "--reasoning-effort", "medium",
                    "--editor-max-output-tokens", "4096", "--api-timeout", "180",
                    "--output", str(runs)], env=worker_env)
    finally:
        worker_env.pop("OPENAI_API_KEY", None)
        display_astra_report()


## 3. Download evidence, even after failure
Open `START_HERE.md` at the ZIP's top level. It shows the selected run, development results, Astra's hypothesis, actual weight change, training probe effects, and restoration checks. Top-level `latest-*` files expose the current report and raw comparisons. `evidence-index.json` lists every file with its SHA-256 hash. Full historical runs, provider records, per-question scores, and source remain under `backend/`. Unknown or unavailable measurements are marked explicitly.


In [ ]:
from datetime import datetime, timezone
inventory = subprocess.check_output([worker_python, '-m', 'pip', 'list', '--format=json'], text=True)
(workspace / 'package-versions.json').write_text(inventory, encoding='utf-8')
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
destination = workspace / f'kirby-evidence-{stamp}.zip'
selected_report = globals().get('astra_report_path')
if selected_report is None or not Path(selected_report).is_file():
    saved_reports = list(runs.glob('*/report.json'))
    if not saved_reports:
        raise RuntimeError('No saved run report exists to export.')
    selected_report = max(saved_reports, key=lambda path: path.stat().st_mtime_ns)
    print('No current attempt report exists; exporting previous saved evidence:', selected_report)
subprocess.run([worker_python, '-m', 'kirby.evidence', '--backend', str(backend),
                '--report', str(selected_report), '--output', str(destination)], cwd=backend, check=True)
print('Open START_HERE.md inside the ZIP for results and evidence links.')
print('Downloading:', destination.name)
files.download(str(destination))


## Interpreting the result
`Astra candidate evaluated and rolled back` means execution completed, not that the edit helped. Inspect `comparison.signal`, `corrected`, `regressed`, and both accuracies. A development improvement is only a signal: regression testing, repeated search accounting, export/reload, and final evaluation remain future work. `Astra abstained` applies no edit. Rejected proposals and provider errors retain incomplete evidence.

Official references: [GPT-6 Astra](https://developers.openai.com/api/docs/models/gpt-6-astra), [Structured Outputs](https://developers.openai.com/api/docs/guides/structured-outputs?api-mode=responses), [Responses API](https://developers.openai.com/api/reference/python/resources/responses/methods/create).
